## 03 - Gold Aggregation Pipeline
Creates aggregated gold tables for the fraud analytics dashboard.

**Source:** `jrvs_databricks.silver`  
**Target:** `jrvs_databricks.gold`  

**Gold tables created:**
1. `fraud_by_day_of_week` - Fraudulent transactions by day of week
2. `fraud_rate_trend` - Daily fraud rate trend
3. `top_fraudulent_users` - Users with most flagged transactions
4. `user_amount_spikes` - Users with sharp rises vs weekly avg
5. `fraud_rate_by_merchant_category` - Fraud rate per MCC
6. `merchants_high_fraud` - Merchants with unusually high fraud
7. `fraud_by_time_of_day` - Fraud distribution by time period
8. `avg_amount_fraud_vs_legit` - Avg amounts: fraud vs non-fraud
9. `top_fraud_amount_by_category` - Highest total fraud $ by MCC
10. `daily_fraud_losses` - Daily monetary losses from fraud
11. `weekly_unique_fraud_users` - Unique fraudulent users per week
12. `monthly_fraud_spikes` - Monthly fraud patterns
13. `user_behavior_around_fraud` - Before vs after fraud behavior
14. `fraud_by_amount_tier` - Fraud rates by purchase value tier

In [0]:
from pyspark.sql.functions import (
    col, count, sum, avg, when, lit, desc, asc, round as spark_round,
    weekofyear, date_trunc, to_date, datediff, row_number, lag, lead,
    percentile_approx, stddev, min as spark_min, max as spark_max,
    concat_ws, collect_list, expr, coalesce, year, month, dayofweek,
    date_format, hour
)
from pyspark.sql.window import Window

CATALOG = "jrvs_databricks"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

# Read silver tables
txn = spark.table(f"{SILVER}.transactions")
users = spark.table(f"{SILVER}.users")
cards = spark.table(f"{SILVER}.cards")

In [0]:
# Q1: Which day(s) of the week sees the highest number of fraudulent transactions?
gold_fraud_by_dow = (txn
    .filter(col("is_fraud") == 1)
    .groupBy("day_of_week", "day_name")
    .agg(
        count("*").alias("fraud_count"),
        spark_round(avg("amount"), 2).alias("avg_fraud_amount"),
        spark_round(sum("amount"), 2).alias("total_fraud_amount")
    )
    .orderBy("day_of_week")
)

gold_fraud_by_dow.write.mode("overwrite").saveAsTable(f"{GOLD}.fraud_by_day_of_week")
print("1. fraud_by_day_of_week saved")

1. fraud_by_day_of_week saved


In [0]:
# Q2: What is the trend of the fraud rate over the past month?
gold_fraud_rate_trend = (txn
    .withColumn("txn_date", to_date("transaction_date"))
    .groupBy("txn_date")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_transactions"),
        spark_round(sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)), 2).alias("fraud_amount")
    )
    .withColumn("fraud_rate", spark_round(col("fraud_transactions") / col("total_transactions") * 100, 4))
    .orderBy("txn_date")
)

gold_fraud_rate_trend.write.mode("overwrite").saveAsTable(f"{GOLD}.fraud_rate_trend")
print("2. fraud_rate_trend saved")

2. fraud_rate_trend saved


In [0]:
# Q3: Which users have the largest number of flagged (is_fraud=1) transactions?
gold_top_fraud_users = (txn
    .filter(col("is_fraud") == 1)
    .groupBy("client_id")
    .agg(
        count("*").alias("fraud_count"),
        spark_round(sum("amount"), 2).alias("total_fraud_amount"),
        spark_round(avg("amount"), 2).alias("avg_fraud_amount"),
        spark_min("transaction_date").alias("first_fraud_date"),
        spark_max("transaction_date").alias("last_fraud_date")
    )
    .join(users.select("id", "current_age", "gender", "credit_score"),
          col("client_id") == users["id"], "left")
    .drop("id")
    .orderBy(desc("fraud_count"))
)

gold_top_fraud_users.write.mode("overwrite").saveAsTable(f"{GOLD}.top_fraudulent_users")
print("3. top_fraudulent_users saved")

3. top_fraudulent_users saved


In [0]:
# Q4: Are there any users showing a sharp rise in transaction amount compared to weekly average?
weekly_window = Window.partitionBy("client_id").orderBy("txn_week")

user_weekly = (txn
    .withColumn("txn_week", date_trunc("week", "transaction_date"))
    .groupBy("client_id", "txn_week")
    .agg(
        count("*").alias("weekly_txn_count"),
        spark_round(sum("amount"), 2).alias("weekly_total"),
        spark_round(avg("amount"), 2).alias("weekly_avg_amount")
    )
)

gold_user_spikes = (user_weekly
    .withColumn("prev_week_avg", lag("weekly_avg_amount", 1).over(weekly_window))
    .withColumn("prev_4wk_avg",
        avg("weekly_avg_amount").over(
            weekly_window.rowsBetween(-4, -1)
        )
    )
    .filter(col("prev_4wk_avg").isNotNull())
    .withColumn("spike_ratio", spark_round(col("weekly_avg_amount") / col("prev_4wk_avg"), 2))
    .filter(col("spike_ratio") > 3)  # Spike = 3x the 4-week rolling avg
    .orderBy(desc("spike_ratio"))
)

gold_user_spikes.write.mode("overwrite").saveAsTable(f"{GOLD}.user_amount_spikes")
print("4. user_amount_spikes saved")

4. user_amount_spikes saved


In [0]:
# Q5: Which merchant categories exhibit the highest fraud rate?
gold_fraud_by_mcc = (txn
    .groupBy("mcc", "mcc_description")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)), 2).alias("total_fraud_amount")
    )
    .withColumn("fraud_rate", spark_round(col("fraud_count") / col("total_transactions") * 100, 4))
    .orderBy(desc("fraud_rate"))
)

gold_fraud_by_mcc.write.mode("overwrite").saveAsTable(f"{GOLD}.fraud_rate_by_merchant_category")
print("5. fraud_rate_by_merchant_category saved")

5. fraud_rate_by_merchant_category saved


In [0]:
# Q6: Are there specific merchants with unusually high fraud volume?
merchant_fraud = (txn
    .groupBy("merchant_id", "merchant_city", "merchant_state", "mcc", "mcc_description")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)), 2).alias("total_fraud_amount")
    )
    .withColumn("fraud_rate", spark_round(col("fraud_count") / col("total_transactions") * 100, 4))
)

# Flag merchants with fraud rate significantly above the global average
global_fraud_rate = txn.filter(col("is_fraud") == 1).count() / txn.count() * 100

gold_merchants_high_fraud = (merchant_fraud
    .filter(col("fraud_count") > 0)
    .withColumn("global_fraud_rate", lit(round(global_fraud_rate, 4)))
    .withColumn("is_high_fraud", col("fraud_rate") > global_fraud_rate * 2)
    .orderBy(desc("fraud_count"))
)

gold_merchants_high_fraud.write.mode("overwrite").saveAsTable(f"{GOLD}.merchants_high_fraud")
print("6. merchants_high_fraud saved")

6. merchants_high_fraud saved


In [0]:
# Q7: How does fraud distribution vary by time of day (morning vs night)?
gold_fraud_by_time = (txn
    .groupBy("time_of_day", "hour_of_day")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(avg(when(col("is_fraud") == 1, col("amount"))), 2).alias("avg_fraud_amount")
    )
    .withColumn("fraud_rate", spark_round(col("fraud_count") / col("total_transactions") * 100, 4))
    .orderBy("hour_of_day")
)

gold_fraud_by_time.write.mode("overwrite").saveAsTable(f"{GOLD}.fraud_by_time_of_day")
print("7. fraud_by_time_of_day saved")

7. fraud_by_time_of_day saved


In [0]:
# Q8: What's the average transaction amount for fraud vs non-fraud transactions?
gold_avg_amount = (txn
    .groupBy("is_fraud")
    .agg(
        count("*").alias("transaction_count"),
        spark_round(avg("amount"), 2).alias("avg_amount"),
        spark_round(sum("amount"), 2).alias("total_amount"),
        spark_round(percentile_approx("amount", 0.5), 2).alias("median_amount"),
        spark_round(spark_min("amount"), 2).alias("min_amount"),
        spark_round(spark_max("amount"), 2).alias("max_amount"),
        spark_round(stddev("amount"), 2).alias("stddev_amount")
    )
    .withColumn("fraud_label", when(col("is_fraud") == 1, "Fraud").otherwise("Legitimate"))
)

gold_avg_amount.write.mode("overwrite").saveAsTable(f"{GOLD}.avg_amount_fraud_vs_legit")
print("8. avg_amount_fraud_vs_legit saved")

8. avg_amount_fraud_vs_legit saved


In [0]:
# Q9: Which merchant category has the highest total fraud amount?
gold_top_fraud_mcc = (txn
    .filter(col("is_fraud") == 1)
    .groupBy("mcc", "mcc_description")
    .agg(
        count("*").alias("fraud_count"),
        spark_round(sum("amount"), 2).alias("total_fraud_amount"),
        spark_round(avg("amount"), 2).alias("avg_fraud_amount")
    )
    .orderBy(desc("total_fraud_amount"))
)

gold_top_fraud_mcc.write.mode("overwrite").saveAsTable(f"{GOLD}.top_fraud_amount_by_category")
print("9. top_fraud_amount_by_category saved")

9. top_fraud_amount_by_category saved


In [0]:
# Q10: What are the total monetary losses due to fraud each day?
gold_daily_losses = (txn
    .filter(col("is_fraud") == 1)
    .withColumn("txn_date", to_date("transaction_date"))
    .groupBy("txn_date")
    .agg(
        count("*").alias("fraud_count"),
        spark_round(sum("amount"), 2).alias("daily_fraud_loss"),
        spark_round(avg("amount"), 2).alias("avg_fraud_amount")
    )
    .orderBy("txn_date")
)

gold_daily_losses.write.mode("overwrite").saveAsTable(f"{GOLD}.daily_fraud_losses")
print("10. daily_fraud_losses saved")

10. daily_fraud_losses saved


In [0]:
# Q11: How many unique users commit fraudulent transactions per week?
from pyspark.sql.functions import countDistinct

gold_weekly_fraud_users = (txn
    .filter(col("is_fraud") == 1)
    .withColumn("txn_week", date_trunc("week", "transaction_date"))
    .groupBy("txn_week")
    .agg(
        countDistinct("client_id").alias("unique_fraud_users"),
        count("*").alias("total_fraud_txns"),
        spark_round(sum("amount"), 2).alias("weekly_fraud_amount")
    )
    .orderBy("txn_week")
)

gold_weekly_fraud_users.write.mode("overwrite").saveAsTable(f"{GOLD}.weekly_unique_fraud_users")
print("11. weekly_unique_fraud_users saved")

11. weekly_unique_fraud_users saved


In [0]:
# Q12: Do fraud patterns show seasonal or monthly spikes?
gold_monthly_fraud = (txn
    .withColumn("txn_year", year("transaction_date"))
    .withColumn("txn_month", month("transaction_date"))
    .withColumn("year_month", date_format("transaction_date", "yyyy-MM"))
    .groupBy("txn_year", "txn_month", "year_month")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)), 2).alias("monthly_fraud_amount")
    )
    .withColumn("fraud_rate", spark_round(col("fraud_count") / col("total_transactions") * 100, 4))
    .orderBy("txn_year", "txn_month")
)

gold_monthly_fraud.write.mode("overwrite").saveAsTable(f"{GOLD}.monthly_fraud_spikes")
print("12. monthly_fraud_spikes saved")

12. monthly_fraud_spikes saved


In [0]:
# Q13: How has user behavior changed before versus after a fraudulent event?
first_fraud = (txn
    .filter(col("is_fraud") == 1)
    .groupBy("client_id")
    .agg(spark_min("transaction_date").alias("first_fraud_date"))
)

txn_with_fraud_date = txn.join(first_fraud, on="client_id", how="inner")

gold_behavior = (txn_with_fraud_date
    .withColumn("days_from_fraud", datediff(col("transaction_date"), col("first_fraud_date")))
    .withColumn("period",
        when(col("days_from_fraud").between(-30, -1), "30d Before Fraud")
        .when(col("days_from_fraud") == 0, "Day of Fraud")
        .when(col("days_from_fraud").between(1, 30), "30d After Fraud")
    )
    .filter(col("period").isNotNull())
    .groupBy("period")
    .agg(
        count("*").alias("transaction_count"),
        spark_round(avg("amount"), 2).alias("avg_amount"),
        spark_round(sum("amount"), 2).alias("total_amount"),
        countDistinct("client_id").alias("unique_users"),
        spark_round(avg(when(col("is_fraud") == 1, 1).otherwise(0)) * 100, 4).alias("fraud_pct")
    )
)

gold_behavior.write.mode("overwrite").saveAsTable(f"{GOLD}.user_behavior_around_fraud")
print("13. user_behavior_around_fraud saved")

13. user_behavior_around_fraud saved


In [0]:
# Q14: Are fraudulent transactions more common on high-value vs low-value purchases?
gold_fraud_by_tier = (txn
    .withColumn("amount_tier",
        when(col("amount") < 0, "Negative/Refund")
        .when(col("amount") < 25, "$0-$25 (Low)")
        .when(col("amount") < 100, "$25-$100 (Medium)")
        .when(col("amount") < 500, "$100-$500 (High)")
        .when(col("amount") < 1000, "$500-$1K (Very High)")
        .otherwise("$1K+ (Premium)")
    )
    .groupBy("amount_tier")
    .agg(
        count("*").alias("total_transactions"),
        count(when(col("is_fraud") == 1, 1)).alias("fraud_count"),
        spark_round(sum("amount"), 2).alias("total_amount"),
        spark_round(sum(when(col("is_fraud") == 1, col("amount")).otherwise(0)), 2).alias("fraud_amount")
    )
    .withColumn("fraud_rate", spark_round(col("fraud_count") / col("total_transactions") * 100, 4))
    .orderBy("amount_tier")
)

gold_fraud_by_tier.write.mode("overwrite").saveAsTable(f"{GOLD}.fraud_by_amount_tier")
print("14. fraud_by_amount_tier saved")

14. fraud_by_amount_tier saved


In [0]:
# Summary of all gold tables
print("=" * 60)
print("GOLD AGGREGATION COMPLETE")
print("=" * 60)

gold_tables = [
    "fraud_by_day_of_week", "fraud_rate_trend", "top_fraudulent_users",
    "user_amount_spikes", "fraud_rate_by_merchant_category", "merchants_high_fraud",
    "fraud_by_time_of_day", "avg_amount_fraud_vs_legit", "top_fraud_amount_by_category",
    "daily_fraud_losses", "weekly_unique_fraud_users", "monthly_fraud_spikes",
    "user_behavior_around_fraud", "fraud_by_amount_tier"
]

for i, table in enumerate(gold_tables, 1):
    count = spark.table(f"{GOLD}.{table}").count()
    print(f"  {i:2d}. {GOLD}.{table}: {count:,} rows")
print("=" * 60)
print(f"\nAll {len(gold_tables)} gold tables ready for dashboard.")

GOLD AGGREGATION COMPLETE
   1. jrvs_databricks.gold.fraud_by_day_of_week: 7 rows
   2. jrvs_databricks.gold.fraud_rate_trend: 3,591 rows
   3. jrvs_databricks.gold.top_fraudulent_users: 1,196 rows
   4. jrvs_databricks.gold.user_amount_spikes: 4,957 rows
   5. jrvs_databricks.gold.fraud_rate_by_merchant_category: 109 rows
   6. jrvs_databricks.gold.merchants_high_fraud: 1,227 rows
   7. jrvs_databricks.gold.fraud_by_time_of_day: 24 rows
   8. jrvs_databricks.gold.avg_amount_fraud_vs_legit: 2 rows
   9. jrvs_databricks.gold.top_fraud_amount_by_category: 98 rows
  10. jrvs_databricks.gold.daily_fraud_losses: 1,571 rows
  11. jrvs_databricks.gold.weekly_unique_fraud_users: 326 rows
  12. jrvs_databricks.gold.monthly_fraud_spikes: 118 rows
  13. jrvs_databricks.gold.user_behavior_around_fraud: 3 rows
  14. jrvs_databricks.gold.fraud_by_amount_tier: 6 rows

All 14 gold tables ready for dashboard.
